# QUESTÕES DE IMPLEMENTAÇÃO DE CÓDIGO AC01

In [88]:
# ============================================================
# QUESTÃO 5 — MULTIGRAFO COM VETOR + LISTAS ENCADEADAS SIMPLES
# ============================================================

import random


class NodoAresta:
    def __init__(self, id_aresta, destino, custo=1.0, caracteristica=""):
        self.id_aresta = id_aresta
        self.destino = destino
        self.custo = custo
        self.caracteristica = caracteristica
        self.prox = None


class ListaEncadeadaArestas:
    def __init__(self):
        self.head = None

    def inserir(self, id_aresta, destino, custo=1.0, caracteristica=""):
        novo = NodoAresta(
            id_aresta,
            destino,
            custo,
            caracteristica
        )
        novo.prox = self.head
        self.head = novo

    def buscar(self, id_aresta):
        atual = self.head

        while atual:
            if atual.id_aresta == id_aresta:
                return atual
            atual = atual.prox

        return None

    def alterar(
        self,
        id_aresta,
        novo_custo=None,
        nova_caracteristica=None
    ):
        no = self.buscar(id_aresta)

        if no:
            if novo_custo is not None:
                no.custo = novo_custo

            if nova_caracteristica is not None:
                no.caracteristica = nova_caracteristica

            return True

        return False

    def remover(self, id_aresta):
        atual = self.head
        anterior = None

        while atual:
            if atual.id_aresta == id_aresta:
                if anterior is None:
                    self.head = atual.prox
                else:
                    anterior.prox = atual.prox

                return True

            anterior = atual
            atual = atual.prox

        return False

    def remover_por_destino(self, destino_id):
        atual = self.head
        anterior = None

        while atual:
            if atual.destino == destino_id:

                if anterior is None:
                    self.head = atual.prox
                else:
                    anterior.prox = atual.prox

                atual = self.head if anterior is None else anterior.prox

            else:
                anterior = atual
                atual = atual.prox

    def para_lista(self):
        lista = []
        atual = self.head

        while atual:
            lista.append(atual)
            atual = atual.prox

        return lista


class Vertice:
    def __init__(
        self,
        id_vertice,
        rotulo="",
        custo=0.0,
        caracteristica=""
    ):
        self.id_vertice = id_vertice
        self.rotulo = rotulo
        self.custo = custo
        self.caracteristica = caracteristica
        self.lista_adj = ListaEncadeadaArestas()


class Multigrafo:
    def __init__(
        self,
        nome="Multigrafo",
        direcionado=True
    ):
        self.nome = nome
        self.direcionado = direcionado

        # vetor de vértices
        self.vertices = []

        # cadastro das arestas para evitar contagem duplicada
        self.arestas = {}

        self.proximo_id_aresta = 1

    # ------------------------------------------------------------
    # CONSTRUIR / DESTRUIR
    # ------------------------------------------------------------

    def destruir(self):
        self.vertices.clear()
        self.arestas.clear()
        self.proximo_id_aresta = 1

    # ------------------------------------------------------------
    # VÉRTICES
    # ------------------------------------------------------------

    def incluir_vertice(
        self,
        id_vertice,
        rotulo="",
        custo=0.0,
        caracteristica=""
    ):
        if self.buscar_vertice(id_vertice) is not None:
            return False

        novo = Vertice(
            id_vertice,
            rotulo,
            custo,
            caracteristica
        )

        self.vertices.append(novo)

        return True

    def buscar_vertice(self, id_vertice):
        for v in self.vertices:
            if v.id_vertice == id_vertice:
                return v

        return None

    def alterar_vertice(
        self,
        id_vertice,
        novo_rotulo=None,
        novo_custo=None,
        nova_caracteristica=None
    ):
        v = self.buscar_vertice(id_vertice)

        if v is None:
            return False

        if novo_rotulo is not None:
            v.rotulo = novo_rotulo

        if novo_custo is not None:
            v.custo = novo_custo

        if nova_caracteristica is not None:
            v.caracteristica = nova_caracteristica

        return True

    def remover_vertice(self, id_vertice):
        v = self.buscar_vertice(id_vertice)

        if v is None:
            return False

        # remove todas as arestas incidentes
        arestas_para_remover = []

        for id_aresta, dados in self.arestas.items():
            origem = dados["origem"]
            destino = dados["destino"]

            if origem == id_vertice or destino == id_vertice:
                arestas_para_remover.append(id_aresta)

        for id_aresta in arestas_para_remover:
            self.remover_aresta(id_aresta)

        self.vertices.remove(v)

        return True

    # ------------------------------------------------------------
    # ARESTAS / LIGAÇÕES
    # ------------------------------------------------------------

    def incluir_aresta(
        self,
        origem_id,
        destino_id,
        custo=1.0,
        caracteristica="",
        id_aresta=None,
        direcionada=None
    ):
        origem = self.buscar_vertice(origem_id)
        destino = self.buscar_vertice(destino_id)

        if origem is None or destino is None:
            return None

        if direcionada is None:
            direcionada = self.direcionado

        if id_aresta is None:
            id_aresta = self.proximo_id_aresta
            self.proximo_id_aresta += 1

        if id_aresta in self.arestas:
            return None

        self.arestas[id_aresta] = {
            "origem": origem_id,
            "destino": destino_id,
            "custo": custo,
            "caracteristica": caracteristica,
            "direcionada": direcionada
        }

        origem.lista_adj.inserir(
            id_aresta,
            destino_id,
            custo,
            caracteristica
        )

        # caso seja um elo não-direcionado, registra nos dois lados
        if not direcionada and origem_id != destino_id:
            destino.lista_adj.inserir(
                id_aresta,
                origem_id,
                custo,
                caracteristica
            )

        return id_aresta

    def buscar_aresta(self, id_aresta):
        if id_aresta not in self.arestas:
            return None, None

        dados = self.arestas[id_aresta]

        origem = self.buscar_vertice(dados["origem"])

        if origem is None:
            return None, None

        aresta = origem.lista_adj.buscar(id_aresta)

        return aresta, dados["origem"]

    def alterar_aresta(
        self,
        id_aresta,
        novo_custo=None,
        nova_caracteristica=None
    ):
        if id_aresta not in self.arestas:
            return False

        dados = self.arestas[id_aresta]

        if novo_custo is not None:
            dados["custo"] = novo_custo

        if nova_caracteristica is not None:
            dados["caracteristica"] = nova_caracteristica

        # atualiza todas as ocorrências da aresta
        for v in self.vertices:
            no = v.lista_adj.buscar(id_aresta)

            if no:
                if novo_custo is not None:
                    no.custo = novo_custo

                if nova_caracteristica is not None:
                    no.caracteristica = nova_caracteristica

        return True

    def remover_aresta(self, id_aresta):
        if id_aresta not in self.arestas:
            return False

        for v in self.vertices:
            v.lista_adj.remover(id_aresta)

        del self.arestas[id_aresta]

        return True

    # ------------------------------------------------------------
    # MOSTRAR
    # ------------------------------------------------------------

    def mostrar(self):
        print("\n" + "=" * 75)
        print(f"GRAFO: {self.nome}")
        print("=" * 75)

        print(f"Vértices: {len(self.vertices)}")
        print(f"Arestas:  {len(self.arestas)}")
        print(
            f"Tipo: {'Direcionado' if self.direcionado else 'Não direcionado'}"
        )

        for v in self.vertices:

            print(
                f"\nVértice {v.id_vertice} | "
                f"Rótulo: {v.rotulo} | "
                f"Custo: {v.custo} | "
                f"Característica: {v.caracteristica}"
            )

            lista = v.lista_adj.para_lista()

            if not lista:
                print("   Sem ligações.")

            for a in lista:
                print(
                    f"   Aresta {a.id_aresta} -> "
                    f"{a.destino} | "
                    f"Custo: {a.custo} | "
                    f"Característica: {a.caracteristica}"
                )

    # ------------------------------------------------------------
    # PREENCHIMENTO AUTOMÁTICO
    # ------------------------------------------------------------

    def preenchimento_automatico(
        self,
        num_vertices=10,
        num_arestas=15
    ):
        self.destruir()

        tipos_vertices = [
            "Servidor",
            "Roteador",
            "Switch",
            "DataCenter"
        ]

        tipos_arestas = [
            "Fibra",
            "Rádio",
            "Cabo",
            "Satélite"
        ]

        for i in range(num_vertices):
            tipo = random.choice(tipos_vertices)

            self.incluir_vertice(
                i,
                f"{tipo}_{i}",
                round(random.uniform(10, 100), 2),
                tipo
            )

        tentativas = 0

        while len(self.arestas) < num_arestas and tentativas < 1000:
            tentativas += 1

            u = random.randint(0, num_vertices - 1)
            v = random.randint(0, num_vertices - 1)

            if u == v:
                continue

            self.incluir_aresta(
                u,
                v,
                round(random.uniform(1, 50), 2),
                random.choice(tipos_arestas)
            )

    # ------------------------------------------------------------
    # ESTATÍSTICAS
    # ------------------------------------------------------------

    def obter_estatisticas(self):
        total_vertices = len(self.vertices)
        total_arestas = len(self.arestas)

        custo_total = sum(
            dados["custo"]
            for dados in self.arestas.values()
        )

        grau_medio_saida = (
            total_arestas / total_vertices
            if total_vertices > 0
            else 0
        )

        return {
            "Vertices": total_vertices,
            "Arestas": total_arestas,
            "Custo Total Arestas": round(custo_total, 2),
            "Grau Médio": round(grau_medio_saida, 2)
        }


# ============================================================
# DOIS GRAFOS DISTINTOS PARA COMPARAÇÃO
# ============================================================

g1 = Multigrafo(
    "Grafo_Manual",
    direcionado=True
)

g1.incluir_vertice(
    1,
    "Roteador_Central",
    150.0,
    "Equipamento"
)

g1.incluir_vertice(
    2,
    "Switch_A",
    80.0,
    "Distribuição"
)

g1.incluir_vertice(
    3,
    "Servidor_Web",
    200.0,
    "Servidor"
)

a1 = g1.incluir_aresta(
    1,
    2,
    10.5,
    "Fibra - principal"
)

a2 = g1.incluir_aresta(
    1,
    2,
    25.0,
    "Rádio - redundante"
)

a3 = g1.incluir_aresta(
    2,
    3,
    5.0,
    "UTP Cat6"
)

g1.alterar_vertice(
    2,
    novo_rotulo="Switch_A_Atualizado",
    novo_custo=95.0,
    nova_caracteristica="Switch Gerenciável"
)

g1.alterar_aresta(
    a2,
    novo_custo=30.0,
    nova_caracteristica="Rádio - backup"
)


g2 = Multigrafo(
    "Grafo_Automatico",
    direcionado=True
)

g2.preenchimento_automatico(
    num_vertices=8,
    num_arestas=14
)


# ============================================================
# GRAFO DA Q5 QUE SERÁ REUTILIZADO NAS Q6, Q7, Q19 E Q20
# ============================================================

def criar_grafo_q5_dag15():
    grafo = Multigrafo(
        "Grafo_Q5_DAG_15",
        direcionado=True
    )

    for i in range(15):
        grafo.incluir_vertice(
            i,
            f"V{i}",
            custo=float(i + 1),
            caracteristica="DAG"
        )

    arestas = [
        (0,1), (0,2),
        (1,3), (1,4),
        (2,5), (2,6),
        (3,7),
        (4,7), (4,8),
        (5,8), (5,9),
        (6,9),
        (7,10),
        (8,10), (8,11),
        (9,11),
        (10,12),
        (11,13),
        (12,14),
        (13,14)
    ]

    for u, v in arestas:
        grafo.incluir_aresta(
            u,
            v,
            custo=1.0,
            caracteristica="Dependência"
        )

    return grafo


g_q5_dag15 = criar_grafo_q5_dag15()


# ============================================================
# TESTE DAS OPERAÇÕES
# ============================================================

print("=== GRAFO 1 ===")
g1.mostrar()

print("\n=== GRAFO 2 ===")
g2.mostrar()

print("\n=== GRAFO DAG DE 15 VÉRTICES ===")
print(g_q5_dag15.obter_estatisticas())

print("\n=== TESTE DE BUSCA ===")
print(g1.buscar_vertice(2))

print("\n=== TESTE DE BUSCA DE ARESTA ===")
print(g1.buscar_aresta(a1))

print("\n=== TESTE DE REMOÇÃO ===")
id_temp = g1.incluir_vertice(
    99,
    "Temporário",
    1.0,
    "Teste"
)

print(
    "Vértice temporário incluído:",
    g1.buscar_vertice(99) is not None
)

g1.remover_vertice(99)

print(
    "Vértice temporário removido:",
    g1.buscar_vertice(99) is None
)

=== GRAFO 1 ===

GRAFO: Grafo_Manual
Vértices: 3
Arestas:  3
Tipo: Direcionado

Vértice 1 | Rótulo: Roteador_Central | Custo: 150.0 | Característica: Equipamento
   Aresta 2 -> 2 | Custo: 30.0 | Característica: Rádio - backup
   Aresta 1 -> 2 | Custo: 10.5 | Característica: Fibra - principal

Vértice 2 | Rótulo: Switch_A_Atualizado | Custo: 95.0 | Característica: Switch Gerenciável
   Aresta 3 -> 3 | Custo: 5.0 | Característica: UTP Cat6

Vértice 3 | Rótulo: Servidor_Web | Custo: 200.0 | Característica: Servidor
   Sem ligações.

=== GRAFO 2 ===

GRAFO: Grafo_Automatico
Vértices: 8
Arestas:  14
Tipo: Direcionado

Vértice 0 | Rótulo: Switch_0 | Custo: 89.05 | Característica: Switch
   Aresta 14 -> 7 | Custo: 30.35 | Característica: Satélite
   Aresta 11 -> 3 | Custo: 25.62 | Característica: Satélite
   Aresta 4 -> 7 | Custo: 6.76 | Característica: Rádio

Vértice 1 | Rótulo: Switch_1 | Custo: 49.41 | Característica: Switch
   Aresta 1 -> 5 | Custo: 36.27 | Característica: Fibra

Vértice 

In [89]:
# ============================================================
# QUESTÃO 6 — IN-ORDER + ROTAÇÃO/ROTULAÇÃO TOPOLÓGICA
# ============================================================

def travessia_in_order(
    grafo,
    inicio_id=0,
    visitado=None,
    sequencia=None
):
    """
    Convenção de in-order para o DAG:
    primeiro descendente -> vértice -> demais descendentes.
    """

    if visitado is None:
        visitado = set()

    if sequencia is None:
        sequencia = []

    if inicio_id in visitado:
        return sequencia

    visitado.add(inicio_id)

    vertice = grafo.buscar_vertice(inicio_id)

    if vertice is None:
        return sequencia

    vizinhos = sorted(
        vertice.lista_adj.para_lista(),
        key=lambda a: a.destino
    )

    # primeiro descendente
    if len(vizinhos) > 0:
        primeiro = vizinhos[0].destino

        if primeiro not in visitado:
            travessia_in_order(
                grafo,
                primeiro,
                visitado,
                sequencia
            )

    # raiz
    sequencia.append(inicio_id)

    # demais descendentes
    for aresta in vizinhos[1:]:
        destino = aresta.destino

        if destino not in visitado:
            travessia_in_order(
                grafo,
                destino,
                visitado,
                sequencia
            )

    return sequencia


def ordenacao_topologica_dfs_q6(grafo):
    estado = {
        v.id_vertice: 0
        for v in grafo.vertices
    }

    pilha = []

    def dfs(v_id):
        estado[v_id] = 1

        vertice = grafo.buscar_vertice(v_id)

        vizinhos = sorted(
            vertice.lista_adj.para_lista(),
            key=lambda a: a.destino
        )

        for aresta in vizinhos:
            destino = aresta.destino

            if estado[destino] == 1:
                raise ValueError(
                    "O grafo possui ciclo e não admite "
                    "ordenação topológica."
                )

            if estado[destino] == 0:
                dfs(destino)

        estado[v_id] = 2
        pilha.append(v_id)

    for v in sorted(grafo.vertices, key=lambda x: x.id_vertice):
        if estado[v.id_vertice] == 0:
            dfs(v.id_vertice)

    return pilha[::-1]


# usa o grafo da questão 5
seq_in_order = []

visitados = set()

for v in sorted(
    g_q5_dag15.vertices,
    key=lambda x: x.id_vertice
):
    if v.id_vertice not in visitados:
        travessia_in_order(
            g_q5_dag15,
            v.id_vertice,
            visitados,
            seq_in_order
        )


ordem_topologica_q6 = ordenacao_topologica_dfs_q6(
    g_q5_dag15
)

rotulacao_q6 = {
    vertice_id: posicao
    for posicao, vertice_id
    in enumerate(ordem_topologica_q6, start=1)
}


print("=== QUESTÃO 6 ===")
print("Grafo utilizado:", g_q5_dag15.nome)
print("Número de vértices:", len(g_q5_dag15.vertices))

print("\nTravessia in-order:")
print(seq_in_order)

print("\nOrdenação topológica:")
print(ordem_topologica_q6)

print("\nRotulação topológica:")
for vertice_id in sorted(rotulacao_q6):
    print(
        f"Vértice {vertice_id:2d} "
        f"-> rótulo {rotulacao_q6[vertice_id]:2d}"
    )

=== QUESTÃO 6 ===
Grafo utilizado: Grafo_Q5_DAG_15
Número de vértices: 15

Travessia in-order:
[14, 12, 10, 7, 3, 1, 4, 8, 13, 11, 0, 5, 9, 2, 6]

Ordenação topológica:
[0, 2, 6, 5, 9, 1, 4, 8, 11, 13, 3, 7, 10, 12, 14]

Rotulação topológica:
Vértice  0 -> rótulo  1
Vértice  1 -> rótulo  6
Vértice  2 -> rótulo  2
Vértice  3 -> rótulo 11
Vértice  4 -> rótulo  7
Vértice  5 -> rótulo  4
Vértice  6 -> rótulo  3
Vértice  7 -> rótulo 12
Vértice  8 -> rótulo  8
Vértice  9 -> rótulo  5
Vértice 10 -> rótulo 13
Vértice 11 -> rótulo  9
Vértice 12 -> rótulo 14
Vértice 13 -> rótulo 10
Vértice 14 -> rótulo 15


In [90]:
#========7=========

# 1. Construção do grafo (DAG) com pelo menos 15 vértices, usando a estrutura da Questão 5
g_topo = Multigrafo("Grafo_Rotulacao_PosOrdem")
for i in range(15):
    g_topo.incluir_vertice(i, f"V{i}")

arestas_dag = [
    (0,1),(0,2),(1,3),(1,4),(2,5),(2,6),
    (3,7),(4,7),(4,8),(5,8),(5,9),(6,9),
    (7,10),(8,10),(8,11),(9,11),(10,12),(11,13),
    (12,14),(13,14)
]
for u, v in arestas_dag:
    g_topo.incluir_aresta(u, v)

# 2. Travessia em pós-ordem: o vértice só é processado DEPOIS de visitar todos os seus descendentes
def dfs_pos_ordem(grafo, inicio_id, visitado, sequencia_pos_ordem):
    visitado.add(inicio_id)
    v = grafo.buscar_vertice(inicio_id)
    for aresta in v.lista_adj.para_lista():
        if aresta.destino not in visitado:
            dfs_pos_ordem(grafo, aresta.destino, visitado, sequencia_pos_ordem)
    sequencia_pos_ordem.append(inicio_id)  # só entra na lista após esgotar os filhos

def rotulacao_topologica_pos_ordem(grafo):
    visitado = set()
    sequencia_pos_ordem = []
    for v in grafo.vertices:
        if v.id_vertice not in visitado:
            dfs_pos_ordem(grafo, v.id_vertice, visitado, sequencia_pos_ordem)
    ordem_topologica = sequencia_pos_ordem[::-1]  # inverter a pós-ordem dá a ordem topológica válida
    rotulos = {vid: pos for pos, vid in enumerate(ordem_topologica)}
    return sequencia_pos_ordem, ordem_topologica, rotulos

# 3. Execução
seq_pos, ordem_topo, rotulos = rotulacao_topologica_pos_ordem(g_topo)

print("=== TRAVESSIA EM PÓS-ORDEM (Questão 7) ===")
print(f"Sequência de finalização (pós-ordem): {seq_pos}\n")
print(f"Ordem topológica obtida (inverso da pós-ordem): {ordem_topo}\n")
print("Rótulos atribuídos aos vértices:")
for v in sorted(rotulos.keys()):
    print(f"  Vértice {v:2d} -> Rótulo Topológico: {rotulos[v]:2d}")

=== TRAVESSIA EM PÓS-ORDEM (Questão 7) ===
Sequência de finalização (pós-ordem): [14, 13, 11, 9, 6, 12, 10, 8, 5, 2, 7, 4, 3, 1, 0]

Ordem topológica obtida (inverso da pós-ordem): [0, 1, 3, 4, 7, 2, 5, 8, 10, 12, 6, 9, 11, 13, 14]

Rótulos atribuídos aos vértices:
  Vértice  0 -> Rótulo Topológico:  0
  Vértice  1 -> Rótulo Topológico:  1
  Vértice  2 -> Rótulo Topológico:  5
  Vértice  3 -> Rótulo Topológico:  2
  Vértice  4 -> Rótulo Topológico:  3
  Vértice  5 -> Rótulo Topológico:  6
  Vértice  6 -> Rótulo Topológico: 10
  Vértice  7 -> Rótulo Topológico:  4
  Vértice  8 -> Rótulo Topológico:  7
  Vértice  9 -> Rótulo Topológico: 11
  Vértice 10 -> Rótulo Topológico:  8
  Vértice 11 -> Rótulo Topológico: 12
  Vértice 12 -> Rótulo Topológico:  9
  Vértice 13 -> Rótulo Topológico: 13
  Vértice 14 -> Rótulo Topológico: 14


In [91]:
#========8=========
# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(10))
ARESTAS = [(0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(0,9),(2,5),(4,8)]

# 2. LÓGICA
def construir_adjacencia(vertices, arestas):
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        adj[u].append(v)
        adj[v].append(u)
    return adj

def subgrafo_maximal_arvore(vertices, arestas):
    """Produz uma árvore geradora de G via DFS: é um subgrafo maximal em forma
    de árvore porque cobre todos os vértices e não dá pra adicionar nenhuma
    aresta de G a ela sem formar ciclo."""
    adj = construir_adjacencia(vertices, arestas)
    visitado = set()
    arestas_arvore = []

    def dfs(u):
        visitado.add(u)
        for v in adj[u]:
            if v not in visitado:
                arestas_arvore.append((u, v))
                dfs(v)

    for v in vertices:
        if v not in visitado:
            dfs(v)  # cobre também grafos desconexos (gera floresta geradora)

    conj_arvore = {tuple(sorted(t)) for t in arestas_arvore}
    arestas_fora = [a for a in arestas if tuple(sorted(a)) not in conj_arvore]
    return arestas_arvore, arestas_fora

# 3. EXECUÇÃO
arvore, fora = subgrafo_maximal_arvore(VERTICES, ARESTAS)

print("=== SUBGRAFO MAXIMAL ÁRVORE (ÁRVORE GERADORA) ===")
print(f"Vértices: {len(VERTICES)} | Arestas originais: {len(ARESTAS)}")
print(f"Arestas da árvore geradora ({len(arvore)}): {arvore}")
print(f"Arestas de G fora da árvore ({len(fora)}): {fora}")
print(f"\nVerificação: |arestas da árvore| = {len(arvore)} (esperado |V|-1 = {len(VERTICES)-1} p/ grafo conexo)")

=== SUBGRAFO MAXIMAL ÁRVORE (ÁRVORE GERADORA) ===
Vértices: 10 | Arestas originais: 13
Arestas da árvore geradora (9): [(0, 1), (1, 3), (3, 2), (2, 5), (5, 4), (4, 8), (8, 7), (7, 6), (8, 9)]
Arestas de G fora da árvore (4): [(0, 2), (3, 4), (5, 6), (0, 9)]

Verificação: |arestas da árvore| = 9 (esperado |V|-1 = 9 p/ grafo conexo)


In [92]:
#========9=========
from collections import defaultdict

# 1. ÁREA DE INPUT EDITÁVEL
ARESTAS_G1 = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(9,0)]  # C10
ARESTAS_G2 = [(10,11),(11,12),(12,13),(13,14),(14,15),(15,16),(16,17),(17,18),(18,19),(19,10)]  # outro C10

# 2. LÓGICA
def construir_adj_set(arestas):
    adj = defaultdict(set)
    vertices = set()
    for u, v in arestas:
        adj[u].add(v); adj[v].add(u)
        vertices.update([u, v])
    return adj, vertices

def grau_sequencia(adj, vertices):
    return sorted(len(adj[v]) for v in vertices)

def sao_isomorfos(arestas_g1, arestas_g2):
    adj1, v1 = construir_adj_set(arestas_g1)
    adj2, v2 = construir_adj_set(arestas_g2)

    # condições necessárias: mesmo nº de vértices/arestas e mesma sequência de graus
    if len(v1) != len(v2) or len(arestas_g1) != len(arestas_g2):
        return False, None, grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)
    if grau_sequencia(adj1, v1) != grau_sequencia(adj2, v2):
        return False, None, grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)

    # busca por bijeção via backtracking com poda por adjacência (viável p/ ~10-20 vértices)
    v1_list, v2_list = list(v1), list(v2)
    mapeamento, usados = {}, set()

    def backtrack(i):
        if i == len(v1_list):
            return True
        u = v1_list[i]
        for cand in v2_list:
            if cand in usados:
                continue
            valido = True
            for w in v1_list[:i]:
                if (w in adj1[u]) != (mapeamento[w] in adj2[cand]):
                    valido = False
                    break
            if valido:
                mapeamento[u] = cand; usados.add(cand)
                if backtrack(i + 1):
                    return True
                usados.remove(cand); del mapeamento[u]
        return False

    if backtrack(0):
        return True, dict(mapeamento), grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)
    return False, None, grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)

# 3. EXECUÇÃO
resultado, mapa, seq1, seq2 = sao_isomorfos(ARESTAS_G1, ARESTAS_G2)

print("=== VERIFICAÇÃO DE ISOMORFISMO ENTRE G1 E G2 ===")
print(f"Sequência de graus G1: {seq1}")
print(f"Sequência de graus G2: {seq2}")
if resultado:
    print("\nResultado: G1 e G2 SÃO ISOMORFOS.")
    print("Mapeamento (bijeção) encontrado:")
    for k in sorted(mapa.keys()):
        print(f"  {k}  ->  {mapa[k]}")
else:
    print("\nResultado: G1 e G2 NÃO são isomorfos.")

=== VERIFICAÇÃO DE ISOMORFISMO ENTRE G1 E G2 ===
Sequência de graus G1: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Sequência de graus G2: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Resultado: G1 e G2 SÃO ISOMORFOS.
Mapeamento (bijeção) encontrado:
  0  ->  10
  1  ->  11
  2  ->  12
  3  ->  13
  4  ->  14
  5  ->  15
  6  ->  16
  7  ->  17
  8  ->  18
  9  ->  19


In [93]:
# ============================================================
# QUESTÃO 10 — JACCARD, COSSENO E SOBREPOSIÇÃO
# ============================================================

import math


# Dois grafos com pelo menos 10 vértices
VERTICES_G1 = list(range(10))
VERTICES_G2 = list(range(10))


ARESTAS_G1 = [
    (0,1), (0,2),
    (1,3), (1,4),
    (2,5), (2,6),
    (3,7), (4,7),
    (5,8), (6,9),
    (7,8), (8,9)
]


ARESTAS_G2 = [
    (0,1), (0,2),
    (1,3), (1,4),
    (2,5), (2,6),
    (3,7), (4,8),
    (5,8), (6,9),
    (7,9), (8,9)
]


def normalizar_arestas(arestas):
    return {
        tuple(sorted((u, v)))
        for u, v in arestas
    }


def calcular_jaccard(arestas1, arestas2):
    A = normalizar_arestas(arestas1)
    B = normalizar_arestas(arestas2)

    uniao = A | B

    if not uniao:
        return 1.0

    return len(A & B) / len(uniao)


def calcular_cosseno(arestas1, arestas2):
    A = normalizar_arestas(arestas1)
    B = normalizar_arestas(arestas2)

    if len(A) == 0 or len(B) == 0:
        return 0.0

    return len(A & B) / math.sqrt(
        len(A) * len(B)
    )


def calcular_sobreposicao(arestas1, arestas2):
    A = normalizar_arestas(arestas1)
    B = normalizar_arestas(arestas2)

    menor = min(len(A), len(B))

    if menor == 0:
        return 0.0

    return len(A & B) / menor


def calcular_metricas_similaridade(
    vertices1,
    arestas1,
    vertices2,
    arestas2
):
    if set(vertices1) != set(vertices2):
        raise ValueError(
            "Para as métricas de arestas, os dois grafos "
            "precisam possuir a mesma identificação de vértices "
            "ou receber um mapeamento explícito."
        )

    jaccard = calcular_jaccard(
        arestas1,
        arestas2
    )

    cosseno = calcular_cosseno(
        arestas1,
        arestas2
    )

    sobreposicao = calcular_sobreposicao(
        arestas1,
        arestas2
    )

    return jaccard, cosseno, sobreposicao


jaccard, cosseno, sobreposicao = (
    calcular_metricas_similaridade(
        VERTICES_G1,
        ARESTAS_G1,
        VERTICES_G2,
        ARESTAS_G2
    )
)


print("=== QUESTÃO 10 ===")

print(f"Vértices G1: {len(VERTICES_G1)}")
print(f"Vértices G2: {len(VERTICES_G2)}")

print(f"\nArestas G1: {len(ARESTAS_G1)}")
print(f"Arestas G2: {len(ARESTAS_G2)}")

print("\nMétricas de similaridade:")
print(f"Jaccard:      {jaccard:.4f}")
print(f"Cosseno:      {cosseno:.4f}")
print(f"Sobreposição: {sobreposicao:.4f}")

=== QUESTÃO 10 ===
Vértices G1: 10
Vértices G2: 10

Arestas G1: 12
Arestas G2: 12

Métricas de similaridade:
Jaccard:      0.7143
Cosseno:      0.8333
Sobreposição: 0.8333


In [94]:
#========11========

from collections import deque

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere seu grafo G(V, E) aqui)
# =====================================================================

# Lista de Vértices
VERTICES = [0, 1, 2, 3, 4, 5, 6, 7]

# Lista de Arestas (Grafos não-direcionados)
ARESTAS = [
    (0, 1), (1, 2), (2, 3), (3, 0),  # Ciclo de tamanho 4 (0-1-2-3-0)
    (2, 4), (4, 5), (5, 6), (6, 2),  # Ciclo de tamanho 4 (2-4-5-6-2)
    (0, 5), (3, 6), (6, 7), (7, 3)   # Adiciona conexões que geram ciclos maiores e menores
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA
# =====================================================================

def construir_lista_adjacencia(vertices, arestas):
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        if v not in adj[u]: adj[u].append(v)
        if u not in adj[v]: adj[v].append(u)
    return adj

def calcular_cintura(adj):
    """Encontra o menor ciclo (Cintura) usando Busca em Largura (BFS)"""
    menor_ciclo = float('inf')

    for inicio in adj:
        distancias = {inicio: 0}
        # Fila armazena: (vertice_atual, vertice_pai)
        fila = deque([(inicio, -1)])

        while fila:
            atual, pai = fila.popleft()

            for vizinho in adj[atual]:
                if vizinho not in distancias:
                    distancias[vizinho] = distancias[atual] + 1
                    fila.append((vizinho, atual))
                elif vizinho != pai:
                    # Encontrou uma aresta cruzada que fecha um ciclo
                    tamanho_ciclo = distancias[atual] + distancias[vizinho] + 1
                    menor_ciclo = min(menor_ciclo, tamanho_ciclo)

    return menor_ciclo if menor_ciclo != float('inf') else None

def calcular_circunferencia(adj):
    """Encontra o maior ciclo (Circunferência) usando DFS com Backtracking"""
    maior_ciclo = 0

    def dfs_backtracking(atual, inicio, visitados, comprimento, pai):
        nonlocal maior_ciclo
        visitados.add(atual)

        for vizinho in adj[atual]:
            # Se o vizinho é o início e não é de onde viemos, fechamos um ciclo
            if vizinho == inicio and vizinho != pai and comprimento >= 3:
                maior_ciclo = max(maior_ciclo, comprimento)
            # Se não visitamos o vizinho nesta rota, continuamos aprofundando
            elif vizinho not in visitados:
                dfs_backtracking(vizinho, inicio, visitados, comprimento + 1, atual)

        visitados.remove(atual) # Backtracking para permitir explorar outras rotas

    for vertice in adj:
        dfs_backtracking(vertice, vertice, set(), 1, -1)

    return maior_ciclo if maior_ciclo >= 3 else None

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

adjacencia = construir_lista_adjacencia(VERTICES, ARESTAS)
cintura = calcular_cintura(adjacencia)
circunferencia = calcular_circunferencia(adjacencia)

print("=== MÉTRICAS DO GRAFO ===")
if cintura is None or circunferencia is None:
    print("O grafo é Acíclico (não possui ciclos).")
    print("Cintura: Infinito")
    print("Circunferência: 0")
else:
    print(f"Cintura (Menor Ciclo): {cintura}")
    print(f"Circunferência (Maior Ciclo): {circunferencia}")

=== MÉTRICAS DO GRAFO ===
Cintura (Menor Ciclo): 3
Circunferência (Maior Ciclo): 8


In [95]:
#==========12===========

from collections import deque

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere o grafo e o vértice alvo aqui)
# =====================================================================

# Lista de Vértices
VERTICES = [0, 1, 2, 3, 4, 5]

# Lista de Arestas
ARESTAS = [
    (0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (4, 5)
]

# Vértice a ser analisado
VERTICE_ALVO = 0


# =====================================================================
# 2. LÓGICA MATEMÁTICA
# =====================================================================

def calcular_excentricidade(vertices, arestas, target):
    # Construção da Lista de Adjacência
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        adj[u].append(v)
        adj[v].append(u)  # Remova esta linha se o grafo for direcionado

    if target not in adj:
        return None, {}

    # BFS a partir do vértice alvo para encontrar as menores distâncias
    distancias = {v: float('inf') for v in vertices}
    distancias[target] = 0
    fila = deque([target])

    while fila:
        atual = fila.popleft()
        for vizinho in adj[atual]:
            if distancias[vizinho] == float('inf'):
                distancias[vizinho] = distancias[atual] + 1
                fila.append(vizinho)

    # A excentricidade é a maior distância encontrada
    excentricidade = max(distancias.values())
    return excentricidade, distancias

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

exc, dists = calcular_excentricidade(VERTICES, ARESTAS, VERTICE_ALVO)

print(f"=== RESULTADO PARA O VÉRTICE {VERTICE_ALVO} ===")
if exc == float('inf'):
    print(f"Excentricidade e({VERTICE_ALVO}): Infinito (O grafo é desconexo ou o vértice não alcança todos os nós).")
else:
    print(f"Excentricidade e({VERTICE_ALVO}): {exc}")

print("\nDistâncias mínimas calculadas:")
for v, d in dists.items():
    print(f"  d({VERTICE_ALVO}, {v}) = {d}")

=== RESULTADO PARA O VÉRTICE 0 ===
Excentricidade e(0): 4

Distâncias mínimas calculadas:
  d(0, 0) = 0
  d(0, 1) = 1
  d(0, 2) = 1
  d(0, 3) = 2
  d(0, 4) = 3
  d(0, 5) = 4


In [96]:
#==============13============

from collections import deque

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere o grafo a ser analisado aqui)
# =====================================================================

# Lista de Vértices
VERTICES = [0, 1, 2, 3, 4, 5]

# Lista de Arestas (Grafo não-direcionado)
ARESTAS = [
    (0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (4, 5)
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA (Não é necessário alterar)
# =====================================================================

def bfs_excentricidade(start, adj, vertices):
    distancias = {v: float('inf') for v in vertices}
    distancias[start] = 0
    fila = deque([start])

    while fila:
        atual = fila.popleft()
        for vizinho in adj[atual]:
            if distancias[vizinho] == float('inf'):
                distancias[vizinho] = distancias[atual] + 1
                fila.append(vizinho)

    return max(distancias.values())

def calcular_propriedades_grafo(vertices, arestas):
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        adj[u].append(v)
        adj[v].append(u)  # Remova esta linha se o grafo for direcionado

    excentricidades = {}
    for v in vertices:
        excentricidades[v] = bfs_excentricidade(v, adj, vertices)

    # Se houver vértices inalcançáveis (grafo desconexo)
    if any(e == float('inf') for e in excentricidades.values()):
        return float('inf'), float('inf'), [], excentricidades

    raio = min(excentricidades.values())
    diametro = max(excentricidades.values())
    centro = [v for v, e in excentricidades.items() if e == raio]

    return raio, diametro, centro, excentricidades

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

raio, diametro, centro, excs = calcular_propriedades_grafo(VERTICES, ARESTAS)

print("=== PROPRIEDADES DO GRAFO G(V, E) ===")
if raio == float('inf'):
    print("O grafo é desconexo (possuindo distâncias infinitas entre vértices).")
else:
    print(f"Raio r(G):     {raio}")
    print(f"Diâmetro d(G): {diametro}")
    print(f"Centro C(G):   {centro}")

print("\nExcentricidades individuais:")
for v, e in excs.items():
    print(f"  e({v}) = {e}")

=== PROPRIEDADES DO GRAFO G(V, E) ===
Raio r(G):     2
Diâmetro d(G): 4
Centro C(G):   [3]

Excentricidades individuais:
  e(0) = 4
  e(1) = 3
  e(2) = 3
  e(3) = 2
  e(4) = 3
  e(5) = 4


In [97]:
#========14=========
# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = [0,1,2,3,4,5,6,7]
ARESTAS = [(0,1),(1,2),(2,0),(1,3),(3,4),(4,5),(5,3),(4,6),(6,7)]

# 2. LÓGICA (Algoritmo de Tarjan: DFS com tempo de descoberta e low-link)
def encontrar_cortes(vertices, arestas):
    adj = construir_adjacencia(vertices, arestas)  # reaproveitando a função da Q8
    descoberta, low = {}, {}
    pai = {v: None for v in vertices}
    visitado = set()
    tempo = [0]
    vertices_corte = set()
    arestas_corte = []

    def dfs(u):
        visitado.add(u)
        descoberta[u] = low[u] = tempo[0]; tempo[0] += 1
        filhos = 0
        for v in adj[u]:
            if v not in visitado:
                filhos += 1
                pai[v] = u
                dfs(v)
                low[u] = min(low[u], low[v])
                if low[v] > descoberta[u]:          # ponte
                    arestas_corte.append((u, v))
                if pai[u] is None and filhos > 1:    # raiz com 2+ filhos
                    vertices_corte.add(u)
                if pai[u] is not None and low[v] >= descoberta[u]:  # articulação
                    vertices_corte.add(u)
            elif v != pai[u]:
                low[u] = min(low[u], descoberta[v])

    for v in vertices:
        if v not in visitado:
            dfs(v)
    return sorted(vertices_corte), arestas_corte

# 3. EXECUÇÃO
v_corte, a_corte = encontrar_cortes(VERTICES, ARESTAS)

print("=== CORTE EM VÉRTICES E ARESTAS DE G(V,E) ===")
print(f"Vértices de corte (pontos de articulação): {v_corte}")
print(f"Arestas de corte (pontes): {a_corte}")

=== CORTE EM VÉRTICES E ARESTAS DE G(V,E) ===
Vértices de corte (pontos de articulação): [1, 3, 4, 6]
Arestas de corte (pontes): [(6, 7), (4, 6), (1, 3)]


In [98]:
#========15=========
from collections import defaultdict, deque

# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(10))
ARESTAS = [(0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(0,9),(2,5),(4,8)]

# 2. LÓGICA
def gerar_arvore_geradora(vertices, arestas):
    adj = construir_adjacencia(vertices, arestas)  # reaproveitando função da Q8
    visitado = set()
    arestas_arvore = []
    def dfs(u):
        visitado.add(u)
        for v in adj[u]:
            if v not in visitado:
                arestas_arvore.append((u, v))
                dfs(v)
    for v in vertices:
        if v not in visitado:
            dfs(v)
    return arestas_arvore

def cortes_fundamentais(vertices, arestas, arvore):
    """Para cada aresta da árvore geradora, remove ela e encontra as duas
    partições de vértices resultantes. O corte fundamental é o conjunto de
    todas as arestas de G que cruzam entre essas duas partições."""
    adj_arvore_base = defaultdict(list)
    for u, v in arvore:
        adj_arvore_base[u].append(v)
        adj_arvore_base[v].append(u)

    resultado = {}
    for (u, v) in arvore:
        adj_temp = defaultdict(list, {k: list(vs) for k, vs in adj_arvore_base.items()})
        adj_temp[u].remove(v)
        adj_temp[v].remove(u)

        # BFS a partir de u na árvore SEM a aresta (u,v) -> acha o lado A
        visitado = {u}
        fila = deque([u])
        while fila:
            atual = fila.popleft()
            for prox in adj_temp[atual]:
                if prox not in visitado:
                    visitado.add(prox)
                    fila.append(prox)
        lado_A = visitado
        lado_B = set(vertices) - lado_A

        corte = [(a, b) for (a, b) in arestas
                 if (a in lado_A and b in lado_B) or (a in lado_B and b in lado_A)]
        resultado[(u, v)] = corte
    return resultado

# 3. EXECUÇÃO
arvore = gerar_arvore_geradora(VERTICES, ARESTAS)
cortes = cortes_fundamentais(VERTICES, ARESTAS, arvore)

print("=== CORTE FUNDAMENTAL DE G(V,E) ===")
print(f"Árvore geradora usada: {arvore}\n")
for aresta_arvore, corte in cortes.items():
    print(f"  Corte fundamental de {aresta_arvore}: {corte}")

=== CORTE FUNDAMENTAL DE G(V,E) ===
Árvore geradora usada: [(0, 1), (1, 3), (3, 2), (2, 5), (5, 4), (4, 8), (8, 7), (7, 6), (8, 9)]

  Corte fundamental de (0, 1): [(0, 1), (0, 2), (0, 9)]
  Corte fundamental de (1, 3): [(0, 2), (1, 3), (0, 9)]
  Corte fundamental de (3, 2): [(0, 2), (2, 3), (3, 4), (0, 9)]
  Corte fundamental de (2, 5): [(3, 4), (0, 9), (2, 5)]
  Corte fundamental de (5, 4): [(3, 4), (4, 5), (5, 6), (0, 9)]
  Corte fundamental de (4, 8): [(5, 6), (0, 9), (4, 8)]
  Corte fundamental de (8, 7): [(5, 6), (7, 8)]
  Corte fundamental de (7, 6): [(5, 6), (6, 7)]
  Corte fundamental de (8, 9): [(8, 9), (0, 9)]


In [99]:
#========16=========
# pip install networkx   (se ainda não tiver instalado)
import networkx as nx

# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(6))
ARESTAS = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3),(4,5)]  # exemplo planar (K4 + aresta solta)

# 2. LÓGICA
def verificar_planaridade(vertices, arestas):
    V, E = len(vertices), len(arestas)
    condicao_euler = (E <= 3*V - 6) if V >= 3 else True  # condição necessária p/ grafo simples

    G = nx.Graph()
    G.add_nodes_from(vertices)
    G.add_edges_from(arestas)
    eh_planar, _ = nx.check_planarity(G)  # teste definitivo (algoritmo Left-Right)
    return eh_planar, condicao_euler, V, E

# 3. EXECUÇÃO
eh_planar, cond_euler, V, E = verificar_planaridade(VERTICES, ARESTAS)

print("=== VERIFICAÇÃO DE PLANARIDADE DE G(V,E) ===")
print(f"Vértices: {V} | Arestas: {E}")
print(f"Condição necessária de Euler (E <= 3V-6): {'OK' if cond_euler else 'VIOLADA'}")
print(f"Resultado do teste de planaridade: {'G É PLANAR' if eh_planar else 'G NÃO É PLANAR'}")

=== VERIFICAÇÃO DE PLANARIDADE DE G(V,E) ===
Vértices: 6 | Arestas: 7
Condição necessária de Euler (E <= 3V-6): OK
Resultado do teste de planaridade: G É PLANAR


In [100]:
#========17=========
# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(15))
ARESTAS = [
    (0,1),(0,2),(1,3),(1,4),(2,5),(2,6),
    (3,7),(4,7),(4,8),(5,8),(5,9),(6,9),
    (7,10),(8,10),(8,11),(9,11),(10,12),(11,13),
    (12,14),(13,14)
]

# 2. LÓGICA
def contrair_aresta(vertices, arestas, aresta_escolhida):
    """Funde v em u: remove a própria aresta contraída (viraria laço),
    redireciona as demais arestas de v para u e elimina arestas duplicadas."""
    u, v = aresta_escolhida
    novos_vertices = [x for x in vertices if x != v]
    novas_arestas = []
    vistas = set()
    for (a, b) in arestas:
        if (a, b) == (u, v) or (a, b) == (v, u):
            continue
        aa = u if a == v else a
        bb = u if b == v else b
        if aa == bb:
            continue  # remove laço resultante
        par = tuple(sorted((aa, bb)))
        if par not in vistas:
            vistas.add(par)
            novas_arestas.append((aa, bb))
    return novos_vertices, novas_arestas

def contracao_maxima(vertices, arestas, verboso=True):
    v_atual, a_atual = list(vertices), list(arestas)
    passo = 1
    while a_atual:
        aresta_escolhida = a_atual[0]
        v_atual, a_atual = contrair_aresta(v_atual, a_atual, aresta_escolhida)
        if verboso:
            print(f"  Passo {passo}: contraiu {aresta_escolhida} -> {len(v_atual)} vértices, {len(a_atual)} arestas restantes")
        passo += 1
    return v_atual, a_atual

# 3. EXECUÇÃO
print("=== CONTRAÇÃO MÁXIMA DE G(V,E) ===")
v_final, a_final = contracao_maxima(VERTICES, ARESTAS)
print(f"\nResultado final: {len(v_final)} vértice(s) restante(s), vértice(s): {v_final}, arestas: {a_final}")

=== CONTRAÇÃO MÁXIMA DE G(V,E) ===
  Passo 1: contraiu (0, 1) -> 14 vértices, 19 arestas restantes
  Passo 2: contraiu (0, 2) -> 13 vértices, 18 arestas restantes
  Passo 3: contraiu (0, 3) -> 12 vértices, 17 arestas restantes
  Passo 4: contraiu (0, 4) -> 11 vértices, 15 arestas restantes
  Passo 5: contraiu (0, 5) -> 10 vértices, 13 arestas restantes
  Passo 6: contraiu (0, 6) -> 9 vértices, 11 arestas restantes
  Passo 7: contraiu (0, 7) -> 8 vértices, 10 arestas restantes
  Passo 8: contraiu (0, 8) -> 7 vértices, 8 arestas restantes
  Passo 9: contraiu (0, 9) -> 6 vértices, 6 arestas restantes
  Passo 10: contraiu (0, 10) -> 5 vértices, 5 arestas restantes
  Passo 11: contraiu (0, 11) -> 4 vértices, 4 arestas restantes
  Passo 12: contraiu (0, 12) -> 3 vértices, 3 arestas restantes
  Passo 13: contraiu (0, 13) -> 2 vértices, 1 arestas restantes
  Passo 14: contraiu (0, 14) -> 1 vértices, 0 arestas restantes

Resultado final: 1 vértice(s) restante(s), vértice(s): [0], arestas: []


In [101]:
# ============================================================
# QUESTÃO 18 — DOIS GRAFOS SIMÉTRICOS, ISOMORFOS
#             E DE CONTRAÇÃO MÁXIMA
# ============================================================

import networkx as nx


def ciclo(vertices):
    arestas = []

    n = len(vertices)

    for i in range(n):
        u = vertices[i]
        v = vertices[(i + 1) % n]
        arestas.append((u, v))

    return arestas


def construir_grafo(vertices, arestas):
    G = nx.Graph()
    G.add_nodes_from(vertices)
    G.add_edges_from(arestas)

    return G


def verificar_simetria_por_rotacao(
    vertices,
    arestas
):
    """
    Verifica a simetria de rotação do ciclo.
    """

    G = construir_grafo(
        vertices,
        arestas
    )

    n = len(vertices)

    mapa = {
        vertices[i]:
            vertices[(i + 1) % n]
        for i in range(n)
    }

    arestas_rotacionadas = {
        frozenset(
            (
                mapa[u],
                mapa[v]
            )
        )
        for u, v in G.edges()
    }

    arestas_originais = {
        frozenset(
            (u, v)
        )
        for u, v in G.edges()
    }

    return (
        arestas_rotacionadas
        == arestas_originais
    )


def contar_automorfismos(G):
    matcher = nx.algorithms.isomorphism.GraphMatcher(
        G,
        G
    )

    return sum(
        1
        for _ in matcher.isomorphisms_iter()
    )


def contrair_aresta(G, u, v):
    H = nx.Graph()

    novos_vertices = [
        x for x in G.nodes()
        if x != v
    ]

    H.add_nodes_from(novos_vertices)

    novo_u = u

    for a, b in G.edges():

        if {a, b} == {u, v}:
            continue

        a2 = novo_u if a == v else a
        b2 = novo_u if b == v else b

        if a2 != b2:
            H.add_edge(a2, b2)

    return H


def contracao_maxima(G):
    H = G.copy()
    passos = []

    while H.number_of_edges() > 0:
        u, v = next(iter(H.edges()))

        H = contrair_aresta(
            H,
            u,
            v
        )

        passos.append(
            (
                (u, v),
                H.number_of_nodes(),
                H.number_of_edges()
            )
        )

    return H, passos


# ------------------------------------------------------------
# G1 = C10
# G2 = C10 com identificadores diferentes
# ------------------------------------------------------------

V1 = list(range(10))
V2 = list(range(10, 20))

E1 = ciclo(V1)
E2 = ciclo(V2)


G1 = construir_grafo(V1, E1)
G2 = construir_grafo(V2, E2)


# ------------------------------------------------------------
# Simetria
# ------------------------------------------------------------

simetria_g1 = verificar_simetria_por_rotacao(
    V1,
    E1
)

simetria_g2 = verificar_simetria_por_rotacao(
    V2,
    E2
)


# ------------------------------------------------------------
# Isomorfismo
# ------------------------------------------------------------

matcher = nx.algorithms.isomorphism.GraphMatcher(
    G1,
    G2
)

eh_isomorfo = matcher.is_isomorphic()

mapeamento = None

if eh_isomorfo:
    mapeamento = next(
        matcher.isomorphisms_iter()
    )


# ------------------------------------------------------------
# Automorfismos
# ------------------------------------------------------------

automorfismos_g1 = contar_automorfismos(G1)
automorfismos_g2 = contar_automorfismos(G2)


# ------------------------------------------------------------
# Contração máxima
# ------------------------------------------------------------

G1_final, passos_g1 = contracao_maxima(G1)
G2_final, passos_g2 = contracao_maxima(G2)


# ------------------------------------------------------------
# RESULTADOS
# ------------------------------------------------------------

print("=== QUESTÃO 18 ===")

print("\nG1:")
print(f"Vértices: {G1.number_of_nodes()}")
print(f"Arestas:  {G1.number_of_edges()}")

print("\nG2:")
print(f"Vértices: {G2.number_of_nodes()}")
print(f"Arestas:  {G2.number_of_edges()}")


print("\n=== SIMETRIA ===")
print(
    "G1 possui simetria por rotação:",
    simetria_g1
)

print(
    "G2 possui simetria por rotação:",
    simetria_g2
)

print(
    "Automorfismos de G1:",
    automorfismos_g1
)

print(
    "Automorfismos de G2:",
    automorfismos_g2
)


print("\n=== ISOMORFISMO ===")
print(
    "G1 e G2 são isomorfos:",
    eh_isomorfo
)

if mapeamento:
    print("\nMapeamento encontrado:")
    for u, v in sorted(
        mapeamento.items()
    ):
        print(f"{u} -> {v}")


print("\n=== CONTRAÇÃO MÁXIMA ===")

print(
    f"G1: "
    f"{G1.number_of_nodes()} -> "
    f"{G1_final.number_of_nodes()} vértice(s)"
)

print(
    f"G2: "
    f"{G2.number_of_nodes()} -> "
    f"{G2_final.number_of_nodes()} vértice(s)"
)

print("\nPassos de contração G1:")
for i, passo in enumerate(
    passos_g1,
    start=1
):
    print(
        f"Passo {i}: "
        f"contraiu {passo[0]} -> "
        f"{passo[1]} vértice(s), "
        f"{passo[2]} aresta(s)"
    )

print("\nPassos de contração G2:")
for i, passo in enumerate(
    passos_g2,
    start=1
):
    print(
        f"Passo {i}: "
        f"contraiu {passo[0]} -> "
        f"{passo[1]} vértice(s), "
        f"{passo[2]} aresta(s)"
    )

=== QUESTÃO 18 ===

G1:
Vértices: 10
Arestas:  10

G2:
Vértices: 10
Arestas:  10

=== SIMETRIA ===
G1 possui simetria por rotação: True
G2 possui simetria por rotação: True
Automorfismos de G1: 20
Automorfismos de G2: 20

=== ISOMORFISMO ===
G1 e G2 são isomorfos: True

Mapeamento encontrado:
0 -> 10
1 -> 11
2 -> 12
3 -> 13
4 -> 14
5 -> 15
6 -> 16
7 -> 17
8 -> 18
9 -> 19

=== CONTRAÇÃO MÁXIMA ===
G1: 10 -> 1 vértice(s)
G2: 10 -> 1 vértice(s)

Passos de contração G1:
Passo 1: contraiu (0, 1) -> 9 vértice(s), 9 aresta(s)
Passo 2: contraiu (0, 9) -> 8 vértice(s), 8 aresta(s)
Passo 3: contraiu (0, 2) -> 7 vértice(s), 7 aresta(s)
Passo 4: contraiu (0, 8) -> 6 vértice(s), 6 aresta(s)
Passo 5: contraiu (0, 3) -> 5 vértice(s), 5 aresta(s)
Passo 6: contraiu (0, 7) -> 4 vértice(s), 4 aresta(s)
Passo 7: contraiu (0, 4) -> 3 vértice(s), 3 aresta(s)
Passo 8: contraiu (0, 6) -> 2 vértice(s), 1 aresta(s)
Passo 9: contraiu (0, 5) -> 1 vértice(s), 0 aresta(s)

Passos de contração G2:
Passo 1: contraiu 

In [102]:
# ============================================================
# QUESTÃO 19 — DFS + ROTAÇÃO/ROTULAÇÃO TOPOLÓGICA
# ============================================================

def obter_adjacencia_direcionada(grafo):
    adj = {
        v.id_vertice: []
        for v in grafo.vertices
    }

    for v in grafo.vertices:
        arestas = sorted(
            v.lista_adj.para_lista(),
            key=lambda a: a.destino
        )

        for aresta in arestas:
            adj[v.id_vertice].append(
                aresta.destino
            )

    return adj


def dfs_travessia(grafo, inicio=0):
    adj = obter_adjacencia_direcionada(grafo)

    visitado = set()
    ordem_dfs = []

    def dfs(v):
        visitado.add(v)
        ordem_dfs.append(v)

        for vizinho in adj[v]:
            if vizinho not in visitado:
                dfs(vizinho)

    dfs(inicio)

    # cobre eventuais componentes restantes
    for v in sorted(adj):
        if v not in visitado:
            dfs(v)

    return ordem_dfs


def ordenacao_topologica_dfs(grafo):
    adj = obter_adjacencia_direcionada(grafo)

    estado = {
        v: 0
        for v in adj
    }

    ordem_final = []

    def dfs(v):
        estado[v] = 1

        for vizinho in adj[v]:

            if estado[vizinho] == 1:
                raise ValueError(
                    "O grafo possui ciclo. "
                    "Não existe ordenação topológica."
                )

            if estado[vizinho] == 0:
                dfs(vizinho)

        estado[v] = 2
        ordem_final.append(v)

    for v in sorted(adj):
        if estado[v] == 0:
            dfs(v)

    return ordem_final[::-1]


# usa diretamente o grafo criado na Q5
grafo_q19 = g_q5_dag15

ordem_dfs = dfs_travessia(
    grafo_q19,
    inicio=0
)

ordem_topologica = ordenacao_topologica_dfs(
    grafo_q19
)

rotulacao = {
    v: i
    for i, v
    in enumerate(
        ordem_topologica,
        start=1
    )
}


print("=== QUESTÃO 19 ===")
print(
    "Grafo utilizado:",
    grafo_q19.nome
)

print(
    "Número de vértices:",
    len(grafo_q19.vertices)
)

print("\nTravessia DFS:")
print(ordem_dfs)

print("\nOrdenação topológica:")
print(ordem_topologica)

print("\nRotulação topológica:")

for v in sorted(rotulacao):
    print(
        f"Vértice {v:2d} "
        f"-> rótulo {rotulacao[v]:2d}"
    )

=== QUESTÃO 19 ===
Grafo utilizado: Grafo_Q5_DAG_15
Número de vértices: 15

Travessia DFS:
[0, 1, 3, 7, 10, 12, 14, 4, 8, 11, 13, 2, 5, 9, 6]

Ordenação topológica:
[0, 2, 6, 5, 9, 1, 4, 8, 11, 13, 3, 7, 10, 12, 14]

Rotulação topológica:
Vértice  0 -> rótulo  1
Vértice  1 -> rótulo  6
Vértice  2 -> rótulo  2
Vértice  3 -> rótulo 11
Vértice  4 -> rótulo  7
Vértice  5 -> rótulo  4
Vértice  6 -> rótulo  3
Vértice  7 -> rótulo 12
Vértice  8 -> rótulo  8
Vértice  9 -> rótulo  5
Vértice 10 -> rótulo 13
Vértice 11 -> rótulo  9
Vértice 12 -> rótulo 14
Vértice 13 -> rótulo 10
Vértice 14 -> rótulo 15


In [103]:
# ============================================================
# QUESTÃO 20 — BFS + ROTAÇÃO/ROTULAÇÃO TOPOLÓGICA
# ============================================================

from collections import deque


def obter_grau_entrada(grafo):
    grau_entrada = {
        v.id_vertice: 0
        for v in grafo.vertices
    }

    for v in grafo.vertices:
        for aresta in v.lista_adj.para_lista():
            destino = aresta.destino

            grau_entrada[destino] += 1

    return grau_entrada


def bfs_travessia(grafo, inicio=0):
    adj = obter_adjacencia_direcionada(
        grafo
    )

    visitado = set()
    fila = deque([inicio])

    ordem_bfs = []

    while fila:
        v = fila.popleft()

        if v in visitado:
            continue

        visitado.add(v)
        ordem_bfs.append(v)

        for vizinho in adj[v]:
            if vizinho not in visitado:
                fila.append(vizinho)

    # cobre componentes restantes
    for v in sorted(adj):
        if v not in visitado:
            fila.append(v)

            while fila:
                atual = fila.popleft()

                if atual in visitado:
                    continue

                visitado.add(atual)
                ordem_bfs.append(atual)

                for vizinho in adj[atual]:
                    if vizinho not in visitado:
                        fila.append(vizinho)

    return ordem_bfs


def ordenacao_topologica_bfs(grafo):
    """
    Algoritmo de Kahn utilizando fila (FIFO).
    """

    adj = obter_adjacencia_direcionada(
        grafo
    )

    grau_entrada = obter_grau_entrada(
        grafo
    )

    fila = deque(
        sorted(
            [
                v
                for v, grau in grau_entrada.items()
                if grau == 0
            ]
        )
    )

    ordem = []

    while fila:
        u = fila.popleft()

        ordem.append(u)

        for v in adj[u]:
            grau_entrada[v] -= 1

            if grau_entrada[v] == 0:
                fila.append(v)

    if len(ordem) != len(adj):
        raise ValueError(
            "O grafo possui ciclo. "
            "Não existe ordenação topológica."
        )

    return ordem


# usa diretamente o grafo criado na Q5
grafo_q20 = g_q5_dag15

ordem_bfs = bfs_travessia(
    grafo_q20,
    inicio=0
)

ordem_topologica_bfs = (
    ordenacao_topologica_bfs(
        grafo_q20
    )
)

rotulacao_bfs = {
    v: i
    for i, v
    in enumerate(
        ordem_topologica_bfs,
        start=1
    )
}


print("=== QUESTÃO 20 ===")
print(
    "Grafo utilizado:",
    grafo_q20.nome
)

print(
    "Número de vértices:",
    len(grafo_q20.vertices)
)

print("\nTravessia BFS:")
print(ordem_bfs)

print("\nOrdenação topológica:")
print(ordem_topologica_bfs)

print("\nRotulação topológica:")

for v in sorted(rotulacao_bfs):
    print(
        f"Vértice {v:2d} "
        f"-> rótulo {rotulacao_bfs[v]:2d}"
    )

=== QUESTÃO 20 ===
Grafo utilizado: Grafo_Q5_DAG_15
Número de vértices: 15

Travessia BFS:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Ordenação topológica:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Rotulação topológica:
Vértice  0 -> rótulo  1
Vértice  1 -> rótulo  2
Vértice  2 -> rótulo  3
Vértice  3 -> rótulo  4
Vértice  4 -> rótulo  5
Vértice  5 -> rótulo  6
Vértice  6 -> rótulo  7
Vértice  7 -> rótulo  8
Vértice  8 -> rótulo  9
Vértice  9 -> rótulo 10
Vértice 10 -> rótulo 11
Vértice 11 -> rótulo 12
Vértice 12 -> rótulo 13
Vértice 13 -> rótulo 14
Vértice 14 -> rótulo 15
